# Étape 4 — Détection d'incohérences + verdict final

Importe `generer_verdict_dossier` depuis `src/detection.py` -- le point
d'entrée unique du pipeline, celui que l'API (v3) appellera directement.

In [6]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

import json
from pathlib import Path
from src.detection import generer_verdict_dossier

OCR_RESULTS_DIR = Path("../outputs/ocr_results")


def charger_ocr(dossier_json_path: Path) -> dict:
    with open(dossier_json_path, "r", encoding="utf-8") as f:
        return json.load(f)


def afficher_verdict(nom_fichier: str, verdict: dict) -> None:
    print(f"=== {nom_fichier} ===")
    print("Cohérent :", verdict["coherent"])
    if verdict["anomalies_champs"]:
        print("  Anomalies de champs :")
        for a in verdict["anomalies_champs"]:
            print(
                "   ", a["champ"], ": page", a["page_reference"], f"({a['valeur_reference']!r})",
                "vs page", a["page_suspecte"], f"({a['valeur_suspecte']!r})",
                f"score={a['score_similarite']:.2f}",
            )
    if verdict["pages_dupliquees"]:
        print("  Pages probablement dupliquées :")
        for d in verdict["pages_dupliquees"]:
            print(
                "    page", d["page_a"], "<-> page", d["page_b"],
                f"(similarité texte={d['similarite_texte']:.2f})",
            )
    print()

## Test sur les 6 dossiers de référence (un par type d'erreur)

In [7]:
for nom_fichier in [
    "Dossier_000001_ocr.json", "Dossier_000005_ocr.json", "Dossier_000022_ocr.json",
    "Dossier_000032_ocr.json", "Dossier_000034_ocr.json", "Dossier_000009_ocr.json",
]:
    chemin = OCR_RESULTS_DIR / nom_fichier
    if not chemin.exists():
        print(f"=== {nom_fichier} === (fichier absent, ignoré)\n")
        continue
    dossier = charger_ocr(chemin)
    verdict = generer_verdict_dossier(dossier["pages"])
    afficher_verdict(nom_fichier, verdict)

=== Dossier_000001_ocr.json ===
Cohérent : True

=== Dossier_000005_ocr.json ===
Cohérent : False
  Anomalies de champs :
    nom_prenom : page 1 ('FASSLElHRL Chaimae') vs page 3 ('OUAHBI Asmaa') score=0.24

=== Dossier_000022_ocr.json ===
Cohérent : False
  Anomalies de champs :
    nom_prenom : page 1 ('ZINE AnOuar') vs page 4 ('BENALI Asmaa') score=0.18

=== Dossier_000032_ocr.json ===
Cohérent : False
  Anomalies de champs :
    date_naissance : page 1 ('05/d5/19B7') vs page 2 ('p2/02/1989') score=0.70

=== Dossier_000034_ocr.json ===
Cohérent : False
  Anomalies de champs :
    numero_dossier : page 1 ('DOS202600034') vs page 3 ('Dos202634735') score=0.67

=== Dossier_000009_ocr.json ===
Cohérent : False
  Pages probablement dupliquées :
    page 4 <-> page 5 (similarité texte=1.00)



## À faire ensuite

Pour ajuster le seuil de détection des doublons, modifie `detecter_doublons` dans `src/detection.py`.